In [1]:
import kagglehub

path = kagglehub.dataset_download("mfekadu/darpa-timit-acousticphonetic-continuous-speech")
print("Dataset path:", path)

100%|██████████| 829M/829M [00:03<00:00, 262MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/mfekadu/darpa-timit-acousticphonetic-continuous-speech/versions/6


In [2]:
!ls -la "{path}/data"

total 16
drwxr-xr-x  4 root root 4096 Nov 21 23:33 .
drwxr-xr-x  3 root root 4096 Nov 21 23:33 ..
drwxr-xr-x 10 root root 4096 Nov 21 23:33 TEST
drwxr-xr-x 10 root root 4096 Nov 21 23:33 TRAIN


In [3]:
!pip install -q faster-whisper soundfile tqdm

In [5]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

In [7]:
import os, glob, ctypes

# Find and preload libcublas.so.12 and libcudnn from pip's nvidia packages
for pattern in [
    "/usr/local/lib/python*/dist-packages/nvidia/cublas/lib/libcublas.so.12",
    "/usr/local/lib/python*/dist-packages/nvidia/cudnn/lib/libcudnn*.so.9",
]:
    for lib_path in glob.glob(pattern):
        ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        print("Loaded:", lib_path)

Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_cnn.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_tensor_ir.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_graph.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_ops.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_heuristic.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_ext.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_precompiled.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_engines_runtime_compiled.so.9
Loaded: /usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib/libcudnn_adv.so.9


In [8]:
import os, glob, json, random
import numpy as np
import soundfile as sf
from tqdm import tqdm
from faster_whisper import WhisperModel

random.seed(42)
np.random.seed(42)

# Point to the TEST directory from your kagglehub `path`
TEST_DIR = os.path.join(path, "data", "TEST")
wrd_files = sorted(glob.glob(os.path.join(TEST_DIR, "**", "*.WRD"), recursive=True))
print(f"Found {len(wrd_files)} TEST utterances")

# Initialize Faster-Whisper on A100
model = WhisperModel("base.en", device="cuda", compute_type="float16")

def load_wrd(wrd_path):
    words = []
    with open(wrd_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                words.append((int(parts[0]), int(parts[1]), parts[2].lower()))
    return words

def corrupt_span(audio, start, end, mode):
    out = audio.copy()
    span = out[start:end]
    if len(span) == 0 or mode == "clean":
        return out
    if mode == "silence":
        out[start:end] = 0.0
        return out

    snr_db = 5.0 if mode == "noise_5db" else -5.0
    sig_power = np.mean(span ** 2) + 1e-10
    noise_power = sig_power / (10 ** (snr_db / 10.0))
    noise = np.random.normal(0, np.sqrt(noise_power), size=span.shape)
    out[start:end] = np.clip(span + noise, -1.0, 1.0)
    return out

def transcribe_array(audio_float32):
    # Faster-whisper accepts 16kHz float32 numpy arrays directly
    segments, _ = model.transcribe(audio_float32, language="en", word_timestamps=True)
    decoded = []
    for seg in segments:
        if seg.words:
            for w in seg.words:
                decoded.append({
                    "word": w.word.strip(),
                    "start": round(w.start, 3),
                    "end": round(w.end, 3),
                    "probability": round(float(w.probability), 4)
                })
    return decoded

results = []
modes = ["clean", "noise_5db", "noise_-5db", "silence"]

for wrd_path in tqdm(wrd_files):
    base = wrd_path[:-4]
    # Kaggle TIMIT includes both .WAV (NIST) and .WAV.wav (RIFF); prefer .WAV.wav if present
    wav_path = base + ".WAV.wav" if os.path.exists(base + ".WAV.wav") else base + ".WAV"
    if not os.path.exists(wav_path):
        continue

    audio, sr = sf.read(wav_path, dtype="float32")
    words = load_wrd(wrd_path)
    if len(words) < 3:
        continue

    # Pick 1 interior target word (avoid first/last word so both W_{t-1} and W_{t+1} exist)
    valid_idxs = [i for i in range(1, len(words) - 1) if len(words[i][2]) >= 3]
    if not valid_idxs:
        continue
    target_idx = random.choice(valid_idxs)
    t_start, t_end, t_word = words[target_idx]

    utt_record = {
        "utterance_id": os.path.relpath(base, TEST_DIR),
        "ground_truth": [w[2] for w in words],
        "target_idx": target_idx,
        "target_word": t_word,
        "target_span_sec": [round(t_start / sr, 3), round(t_end / sr, 3)],
        "runs": {}
    }

    for mode in modes:
        mod_audio = corrupt_span(audio, t_start, t_end, mode)
        utt_record["runs"][mode] = transcribe_array(mod_audio)

    results.append(utt_record)

with open("phase1_outputs.json", "w") as f:
    json.dump(results, f)

print(f"Saved {len(results)} utterances to phase1_outputs.json")

Found 1680 TEST utterances


100%|██████████| 1680/1680 [13:30<00:00,  2.07it/s]


Saved 1677 utterances to phase1_outputs.json


In [9]:
from google.colab import files
files.download("phase1_outputs.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>